In [ ]:
# ============================================================
# 🎮 VIDEO GAME SALES DASHBOARD 
# ============================================================

In [36]:
import pandas as pd
import numpy as np
import panel as pn
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


In [37]:

pn.extension('plotly')

In [38]:
# -------------------------
# GREEN DARK THEME
# -------------------------
pio.templates["green_dark"] = dict(
    layout=dict(
        paper_bgcolor="#0b1f1a",
        plot_bgcolor="#0b1f1a",
        font=dict(color="#00ff9c"),
        colorway=["#00ff9c", "#00e676", "#1de9b6", "#00c853"]
    )
)
pio.templates.default = "green_dark"

In [39]:
# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv("cleaned_vgsales.csv")

In [40]:
df = df.dropna(subset=['total_sales', 'critic_score'])

In [41]:
# -------------------------
# FILTERS
# -------------------------
console_filter = pn.widgets.Select(
    name="Console",
    options=["All"] + sorted(df['console'].dropna().unique().tolist())
)

genre_filter = pn.widgets.MultiChoice(
    name="Genre",
    options=sorted(df['genre'].dropna().unique().tolist())
)

year_filter = pn.widgets.IntRangeSlider(
    name="Year",
    start=int(df['year'].min()),
    end=int(df['year'].max()),
    value=(int(df['year'].min()), int(df['year'].max()))
)

def filter_data(console, genres, year_range):
    dff = df.copy()
    
    if console != "All":
        dff = dff[dff['console'] == console]
    
    if genres:
        dff = dff[dff['genre'].isin(genres)]
    
    dff = dff[
        (dff['year'] >= year_range[0]) &
        (dff['year'] <= year_range[1])
    ]
    
    return dff


In [42]:
def kpis(console, genres, year_range):
    dff = filter_data(console, genres, year_range)

    return pn.Row(
        pn.indicators.Number(name="Total Games", value=len(dff)),
        pn.indicators.Number(name="Total Sales", value=round(dff['total_sales'].sum(), 2)),
        pn.indicators.Number(name="Avg Critic Score", value=round(dff['critic_score'].mean(), 1)),
        
        # ✅ Replace string KPI with Markdown
        pn.pane.Markdown(
            f"### 🎯 Top Genre\n## {dff['genre'].mode()[0] if len(dff)>0 else 'N/A'}"
        )
    )

In [43]:
# ============================================================
# CHARTS (10 TOTAL)
# ============================================================

def chart1(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.bar(
        dff.sort_values('total_sales', ascending=False).head(10),
        x='total_sales', y='title', orientation='h',
        title="Top 10 Games"
    )

def chart2(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.pie(dff, names='genre', values='total_sales',
                  title="Sales by Genre")

def chart3(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.line(dff.groupby('year')['total_sales'].sum().reset_index(),
                   x='year', y='total_sales',
                   title="Sales Trend")

def chart4(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    region = dff[['na_sales','jp_sales','pal_sales','other_sales']].sum().reset_index()
    region.columns = ['Region','Sales']
    return px.bar(region, x='Region', y='Sales', title="Regional Sales")

def chart5(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.scatter(dff, x='critic_score', y='total_sales',
                      size='total_sales', color='genre',
                      title="Critic Score vs Sales")

def chart6(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.bar(
        dff.groupby('console')['total_sales'].sum().reset_index(),
        x='console', y='total_sales',
        title="Console Performance"
    )

def chart7(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.box(dff, x='genre', y='total_sales',
                  title="Sales Distribution by Genre")

def chart8(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.histogram(dff, x='critic_score',
                        title="Critic Score Distribution")

def chart9(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.area(
        dff.groupby('year')['na_sales'].sum().reset_index(),
        x='year', y='na_sales',
        title="NA Sales Trend"
    )

def chart10(console, genres, year_range):
    dff = filter_data(console, genres, year_range)
    return px.bar(
        dff.groupby('publisher')['total_sales'].sum()
        .sort_values(ascending=False).head(10).reset_index(),
        x='publisher', y='total_sales',
        title="Top Publishers"
    )

In [44]:
# ============================================================
# DASHBOARD UI
# ============================================================

template = pn.template.FastListTemplate(
    title="🎮 Video Game Sales Dashboard",
    header_background="#00c97f",

    sidebar=[
        pn.pane.Markdown("## 🎯 Filters"),
        console_filter,
        genre_filter,
        year_filter
    ],

    main=[

        pn.bind(kpis, console_filter, genre_filter, year_filter),

        pn.Tabs(

            ("Overview",
             pn.Column(
                 pn.bind(chart1, console_filter, genre_filter, year_filter),
                 pn.bind(chart2, console_filter, genre_filter, year_filter),
                 pn.bind(chart3, console_filter, genre_filter, year_filter),
                 pn.bind(chart4, console_filter, genre_filter, year_filter)
             )
            ),

            ("Analysis",
             pn.Column(
                 pn.bind(chart5, console_filter, genre_filter, year_filter),
                 pn.bind(chart6, console_filter, genre_filter, year_filter),
                 pn.bind(chart7, console_filter, genre_filter, year_filter),
                 pn.bind(chart8, console_filter, genre_filter, year_filter)
             )
            ),

            ("Insights",
             pn.Column(
                 pn.bind(chart9, console_filter, genre_filter, year_filter),
                 pn.bind(chart10, console_filter, genre_filter, year_filter)
             )
            )
        )
    ]
)





Launching server at http://localhost:59639


2026-04-23 17:39:17.984 200 GET / (127.0.0.1) 3330.27ms
2026-04-23 17:39:18.065 200 GET /static/extensions/panel/bundled/plotlyplot/maplibre-gl@4.4.1/dist/maplibre-gl.css?v=1.8.10 (127.0.0.1) 18.76ms
2026-04-23 17:39:18.072 200 GET /static/js/bokeh-tables.min.js?v=62532f6ff926a41c47def08668ee2cec06b351a8e8a5ca01a21fdc78811c3fc5d6b048deaf949f6abc5c6963b7bbf56ea285d6ada0b7a61191449d0d115bf737 (127.0.0.1) 4.16ms
2026-04-23 17:39:18.110 200 GET /static/extensions/panel/bundled/reactiveesm/es-module-shims@%5E1.10.0/dist/es-module-shims.min.js (127.0.0.1) 42.00ms
2026-04-23 17:39:18.758 200 GET /static/js/bokeh-widgets.min.js?v=aa4c33f38bd3425dafb970d1ce795ccf91e608a9a6d4db35f85f382954d7a91abf76031394f3af6ecb919a1b5dbbfefa2734c7a543132a8f0c0f24a2f9d3237b (127.0.0.1) 688.91ms
2026-04-23 17:39:18.761 200 GET /static/js/bokeh-gl.min.js?v=b803cb051212c1e9f0fba71c62739c0cfb38114322d67a2ace60eee732710b6541d80907d9fcf0afb2f962c368f1db1e30b8800652d9510d1b801727f9669eb2 (127.0.0.1) 691.41ms
2026-04-2

In [ ]:
pn.serve(template, show=True)